In [1]:
import re
import os
import pandas as pd
from datetime import datetime

In [4]:
INPUT = "raw_data.csv"
OUTPUT = "cleaned_data.csv"

In [3]:
# Helper regular expressions
RE_FOLDER = re.compile(r'\\|/')  # split path by both separators
RE_DIGITS_FOLDER = re.compile(r'\d{9,}')  # folder like 20240226013 (>=9 digits)
RE_DATETIME_DMY = re.compile(r'(\d{2}/\d{2}/\d{4}\s*\d{2}:\d{2}:\d{2})')
RE_DATE_DMY_ONLY = re.compile(r'(\d{2}/\d{2}/\d{4})')
RE_YEAR = re.compile(r'((19|20)\d{2})')
# Name with birth year
RE_NAME_WITH_YEAR = re.compile(r'([A-Za-zÀ-ỹ\.\'\-\s]{2,60}?)\s+((19|20)\d{2})', re.UNICODE)
# xray type candidate
RE_XRAY_TYPE_CAND = re.compile(r'([A-Z0-9\-\/]{2,40}(?:\s+[A-Z0-9\-\/]{1,20})*)')

In [5]:
# Helper functions
def get_folder_from_path(path):
    parts = re.split(RE_FOLDER, path)
    for p in parts[::-1]:
        if p and RE_DIGITS_FOLDER.search(p):
            return p
    parts = [p for p in parts if p]
    if len(parts) >= 2:
        return parts[-2]
    return ""

def extract_datetime(text, filepath):
    m = RE_DATETIME_DMY.search(text)
    if m:
        s = m.group(1)
        try:
            dt = datetime.strptime(s.strip(), "%d/%m/%Y %H:%M:%S")
            return dt.isoformat(sep=' ')
        except:
            pass
    m = RE_DATE_DMY_ONLY.search(text)
    if m:
        s = m.group(1)
        try:
            dt = datetime.strptime(s.strip(), "%d/%m/%Y")
            return dt.date().isoformat()
        except:
            pass
    filename = os.path.basename(filepath)
    m = re.search(r'(\d{14})', filename)
    if m:
        try:
            dt = datetime.strptime(m.group(1), "%Y%m%d%H%M%S")
            return dt.isoformat(sep=' ')
        except:
            pass
    m = re.search(r'(\d{8})_?(\d{6})', filename)
    if m:
        try:
            dt = datetime.strptime(m.group(1)+m.group(2), "%Y%m%d%H%M%S")
            return dt.isoformat(sep=' ')
        except:
            pass
    return ""

def extract_patient_name_and_year(text):
    m = RE_NAME_WITH_YEAR.search(text)
    if m:
        name = m.group(1).strip()
        year = m.group(2)
        name = re.sub(r'\b(Series|Image|MA|MR|MRS|MS)\b.*$', '', name, flags=re.IGNORECASE).strip()
        name = re.sub(r'\s{2,}', ' ', name)
        return name, year
    # fallback if year not found
    m = re.search(r'(?:Image\s*\d+\s*)([A-Za-zÀ-ỹ\-\s]{3,40})', text, flags=re.IGNORECASE)
    if m:
        name = m.group(1).strip()
        name = re.sub(r'\s{2,}', ' ', name)
        if len(name.split()) >= 2:
            year = ""
            return name, year
    return "", ""

def extract_xray_type(text, patient_name, folder):
    t = re.sub(r'^\s*TTYT[-\sA-Z0-9]*\s*', '', text, flags=re.IGNORECASE)
    if patient_name:
        idx = t.find(patient_name)
        if idx != -1:
            t_after = t[idx + len(patient_name):]
        else:
            ym = RE_YEAR.search(t)
            if ym:
                t_after = t[ym.end():]
            else:
                t_after = t
    else:
        if folder and folder in t:
            idx = t.find(folder)
            t_after = t[idx + len(folder):]
        else:
            t_after = t

    cut = re.split(r'Unit:|Unit|Pixel|W:|L:|\d{2}/\d{2}/\d{4}', t_after, maxsplit=1, flags=re.IGNORECASE)[0]

    candidates = RE_XRAY_TYPE_CAND.findall(cut)
    filtered = []
    for c in candidates:
        c = c.strip()
        if len(c) < 2:
            continue
        if re.search(r'\b(Series|Image|MA|MR|MS|Admin|Pixel|Unit)\b', c, flags=re.IGNORECASE):
            continue
        if re.search(r'[A-Z]', c):
            filtered.append(c)
    if filtered:
        return filtered[0].strip()
    known = re.search(r'\b(C-SPINE|CERVICAL|CHEST|THORAX|L-SPINE|T-SPINE|KUB|ABDOMEN|PELVIS|KNEE|SHOULDER|AP|LAT)\b', t, flags=re.IGNORECASE)
    if known:
        return known.group(0).upper()
    return ""

def normalize_row(filepath, text):
    folder = get_folder_from_path(filepath)
    dt = extract_datetime(text, filepath)
    patient, birth_year = extract_patient_name_and_year(text)
    xray = extract_xray_type(text, patient, folder)

    patient = patient.strip(' ,;.-') if patient else ""
    xray = xray.strip(' ,;.-') if xray else ""
    return folder, patient, birth_year, xray, dt

In [6]:
# Main processing
try:
    df = pd.read_csv(INPUT, header=None, names=['filepath', 'extracted_text'], dtype=str, keep_default_na=False)
except Exception as e:
    print("Error reading input CSV:", e)

results = []
for i, row in df.iterrows():
    fp = row['filepath'] if pd.notna(row['filepath']) else ""
    txt = row['extracted_text'] if pd.notna(row['extracted_text']) else ""
    folder, patient, birth_year, xray, dt = normalize_row(fp, txt)
    results.append({
        'filepath': fp,
        'folder': folder,
        'patient_name': patient,
        'birth_year': birth_year,
        'xray_type': xray,
        'datetime': dt
    })
    if i % 5000 == 0 and i > 0:
        print(f"Processed {i} rows...")

out = pd.DataFrame(results)
out.to_csv(OUTPUT, index=False)
print("Done! Saved normalized file to:", OUTPUT)

Processed 5000 rows...
Processed 10000 rows...
Processed 15000 rows...
Processed 20000 rows...
Processed 25000 rows...
Processed 30000 rows...
Processed 35000 rows...
Done! Saved normalized file to: cleaned_data.csv
